In [2]:
%pip install seaborn

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("heart.csv")

# Features and target
X = df.drop(columns="target")
y = df["target"]

# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, shuffle=True, random_state=0
)

# Train Decision Tree
from sklearn.tree import DecisionTreeClassifier
dtclf = DecisionTreeClassifier(random_state=42)

dtclf.fit(X_train, y_train)

# Predictions
y_pred = dtclf.predict(X_test)
y_true = y_test

# Accuracy
from sklearn.metrics import accuracy_score
print("Train accuracy:", np.round(accuracy_score(y_train, dtclf.predict(X_train)), 2))
print("Test accuracy:", np.round(accuracy_score(y_true, y_pred), 2))


# ---------------- Random Search ----------------

import time
start = time.time()

# Correct hyperparameter space
hyperparameter_space = {
    'max_depth':[2,3,4,6,8,10,12,15,20],
    'min_samples_leaf':[1,2,4,6,8,10,20,30],
    'min_samples_split':[2,3,4,5,6,8,10]   # FIXED (removed 1)
}

from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(
    dtclf,
    param_distributions=hyperparameter_space,
    n_iter=10,
    scoring="accuracy",
    random_state=0,
    n_jobs=-1,
    cv=10,
    return_train_score=True
)

rs.fit(X_train, y_train)

print("\nOptimal hyperparameter combination (Random Search):", rs.best_params_)
print("Mean cross-validated training accuracy score:", rs.best_score_)

# Test accuracy
y_pred = rs.best_estimator_.predict(X_test)
print("Test accuracy:", np.round(accuracy_score(y_test, y_pred), 2))

end = time.time()
print("Execution time of Random Search (in Seconds):", end - start)


# ---------------- Grid Search ----------------

start = time.time()

from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(
    dtclf,
    param_grid=hyperparameter_space,
    scoring="accuracy",
    n_jobs=-1,
    cv=10,
    return_train_score=True
)

gs.fit(X_train, y_train)

print("\nOptimal hyperparameter combination (Grid Search):", gs.best_params_)
print("Mean cross-validated training accuracy score:", gs.best_score_)

# Test accuracy
y_pred = gs.best_estimator_.predict(X_test)
print("Test accuracy:", np.round(accuracy_score(y_test, y_pred), 2))

end = time.time()
print("Execution time of Grid Search (in Seconds):", end - start)

Train accuracy: 1.0
Test accuracy: 1.0

Optimal hyperparameter combination (Random Search): {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 15}
Mean cross-validated training accuracy score: 0.9891654978962132
Test accuracy: 0.99
Execution time of Random Search (in Seconds): 2.7149999141693115

Optimal hyperparameter combination (Grid Search): {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Mean cross-validated training accuracy score: 0.9978260869565216
Test accuracy: 1.0
Execution time of Grid Search (in Seconds): 161.33199977874756
